# Most recent benchmarks for BRCA dataset

In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval
from bipartite_gnn.graph_building import cosine_similarity_matrix, keep_n_neighbours, dense_to_coo
from bipartite_gnn.train_test import GNNTrainer
from gnn_experiments.mogonet_eval import mogonet_eval

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data and make sure that everything is aligned

In [6]:
null_vals = ["NA"]

mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
meth = pl.read_csv("BRCA_PROCESSED_DATA/meth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)
cnv = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())

assert mrna.columns[1:] == meth.columns[1:] == cnv.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [7]:
# concat all data into one dataframe
mrna_X = mrna[:, 1:].to_numpy().T
meth_X = meth[:, 1:].to_numpy().T
mirna_X = mirna[:, 1:].to_numpy().T
cnv_X = cnv[:, 1:].to_numpy().T

print(mrna_X.shape, meth_X.shape, mirna_X.shape, cnv_X.shape)

# concat after preprocessing
X_list = [mrna_X] # , meth_X, mirna_X, cnv_X]

# concat before preprocessing
X = np.hstack(X_list)

(483, 18975) (483, 9317) (483, 231) (483, 24776)


In [103]:
knn_eval(X, y, select_n_features=True)

500
| KNN | 0.84 +/- 0.01 | 0.82 +/- 0.02 | 0.83 +/- 0.01 |
study.best_value=0.8294639374375246, study.best_params={'n_neighbors': 11, 'n_features': 1859}


## concat before

| Model | ACC | F1 macro | F1 weighted |
| --- | --- | --- | --- | 
| KNN (mrna) | 0.84 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
| KNN (mrna, mirna) | 0.84 +/- 0.02 | 0.83 +/- 0.03 | 0.84 +/- 0.02 |
| KNN (mrna, mirna, meth) | 0.80 +/- 0.04 | 0.78 +/- 0.06 | 0.79 +/- 0.05 |
| KNN (mrna, mirna, meth, cnv) | 0.80 +/- 0.04 | 0.78 +/- 0.05 | 0.79 +/- 0.05 |

## concat after (KNN)

| subset | weighted f1 |
| --- | --- |
| mrna | 0.83 +/- 0.01 |
| mirna | 0.68 +/- 0.04 |
| meth | 0.61 +/- 0.07 |
| cnv | 0.55 +/- 0.03 |
| mrna, mirna | 0.82 +/- 0.03 |
| mrna, mirna, meth | 0.78 +/- 0.03 |
| mrna, mirna, meth, cnv | 0.72 +/- 0.03 |

In [106]:
svm_eval(X, y, select_n_features=True, n_trials=20, n_features_preselect=10000)

Trial 1 / 20
| SVM | 0.82 +/- 0.02 | 0.82 +/- 0.02 | 0.83 +/- 0.02 |
Trial 2 / 20
| SVM | 0.83 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
Trial 3 / 20
Trial 4 / 20
Trial 5 / 20
Trial 6 / 20
Trial 7 / 20
Trial 8 / 20
Trial 9 / 20
Pruning trial
Trial 10 / 20
Trial 11 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
Trial 12 / 20
Pruning trial
Trial 13 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
Trial 14 / 20
Pruning trial
Trial 15 / 20
Trial 16 / 20
Pruning trial
Trial 17 / 20
Pruning trial
Trial 18 / 20
Pruning trial
Trial 19 / 20
Pruning trial
Trial 20 / 20
| LIN SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
study.best_value=0.834053661222406, study.best_params={'C': 0.00965373842048661, 'class_weight': 'balanced', 'n_features': 1770}


{'acc': 0.832323883161512,
 'f1_macro': 0.8283495019959709,
 'f1_weighted': 0.834053661222406,
 'acc_std': 0.009659601422776734,
 'f1_macro_std': 0.010556149693910116,
 'f1_weighted_std': 0.009406892392122062}

In [ ]:
| LIN SVM (mrna) | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
| LIN SVM (all) | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |

In [108]:
xgboost_eval(X, y, n_trials=20, n_features=5000)

0 / 20
ACC: [0.45360825 0.45360825 0.45360825 0.45833333 0.45833333]
F1M: [0.15602837 0.15602837 0.15602837 0.15714286 0.15714286]
F1W: [0.28310302 0.28310302 0.28310302 0.28809524 0.28809524]
| XGBoost | 0.46 +/- 0.00 | 0.16 +/- 0.00 | 0.29 +/- 0.00 |
1 / 20
ACC: [0.84536082 0.89690722 0.87628866 0.85416667 0.875     ]
F1M: [0.82777469 0.90148423 0.87620773 0.87072756 0.85983827]
F1W: [0.84792499 0.89770671 0.87467005 0.8508768  0.8756662 ]
| XGBoost | 0.87 +/- 0.02 | 0.87 +/- 0.02 | 0.87 +/- 0.02 |
2 / 20
Pruning trial
ACC: [0.81443299 0.79381443 0.81443299 0.80208333 0.        ]
F1M: [0.78562843 0.8141908  0.79612403 0.82675803 0.        ]
F1W: [0.81221407 0.79860159 0.81181172 0.79870123 0.        ]
3 / 20
Pruning trial
ACC: [0.82474227 0.80412371 0.81443299 0.83333333 0.        ]
F1M: [0.76158266 0.78914288 0.80304557 0.8151346  0.        ]
F1W: [0.82362017 0.7957014  0.81356722 0.83152298 0.        ]
4 / 20
Pruning trial
ACC: [0.75257732 0.70103093 0.65979381 0.71875    0.       

| XGBoost (default) | 0.80 +/- 0.02 | 0.80 +/- 0.04 | 0.80 +/- 0.03 |  
| XGBoost (tuned, mrna only) | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |  
| XGBoost (tuned, all) | 0.88 +/- 0.03 | 0.87 +/- 0.04 | 0.88 +/- 0.03 |  

In [5]:
mlp_eval(X, y, reg_type=None, n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.81632656 0.73469388 0.875      0.72916669]
[0.73017544 0.79366829 0.73770343 0.87294477 0.73425214]
[0.78425588 0.81714559 0.73337645 0.87027483 0.72971154]
{'acc': 0.790221095085144, 'f1_macro': 0.7737488137137466, 'f1_weighted': 0.7869528559196068, 'acc_std': 0.05424449923276622, 'f1_macro_std': 0.054762249650224734, 'f1_weighted_std': 0.052930947915494395}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.85714287 0.79591835 0.9375     0.77083331]
[0.73028135 0.86745815 0.80430157 0.95072464 0.77475158]
[0.75822184 0.85827691 0.78980158 0.93683575 0.77207543]
{'acc': 0.8232993125915528, 'f1_macro': 0.8255034573952553, 'f1_weighted': 0.8230423024277783, 'acc_std': 0.06684377267029194, 'f1_macro_std': 0.07685449036354856, 'f1_weighted_std': 0.06647508594798308}
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.87755102 0.81632656 0.89583331 

| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.85 +/- 0.05 | 0.87 +/- 0.04 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.87 +/- 0.06 | 0.87 +/- 0.04 | 

In [6]:
mlp_eval(X, y, reg_type="l1", n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.77551019 0.85714287 0.81632656 0.91666669 0.77083331]
[0.72516835 0.85222222 0.82318841 0.93280632 0.77475158]
[0.77097506 0.85705215 0.81307306 0.91469038 0.77207543]
{'acc': 0.8272959232330322, 'f1_macro': 0.8216273766294975, 'f1_weighted': 0.8255732154725489, 'acc_std': 0.054530887642936364, 'f1_macro_std': 0.07042857909655777, 'f1_weighted_std': 0.05464799916236452}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.5276180169037312 < 0.8255732154725489
[0.6122449  0.65306121 0.63265306 0.         0.        ]
[0.37840909 0.47001263 0.41412338 0.         0.        ]
[0.48061224 0.57462379 0.51851312 0.         0.        ]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.5413564643588856 < 0.8255732154725489
[0.65306121 0.63265306 0.67346936 0.         0.        ]
[0.475      0.42490222 0.50767544 0.         0.        ]
[0.56326531 0.51944762 0.5991

| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.03 | 0.85 +/- 0.04 | 0.86 +/- 0.03 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.07 | 0.86 +/- 0.08 | 0.86 +/- 0.07 |  
| MLP l1 reg (mrna only, var) | 0.84 +/- 0.06 | 0.84 +/- 0.07 | 0.84 +/- 0.06 |


{'acc': 0.867463231086731, 'f1_macro': 0.8446657147200625, 'f1_weighted': 0.8687115181280782, 'acc_std': 0.0655361141104709, 'f1_macro_std': 0.09939143723563133, 'f1_weighted_std': 0.05932617979442903}

mrna only study.best_value=2.577811014967633, study.best_params={'lr': 0.00012898079159581874, 'l1_lambda': 0.00047366984296337946, 'l2_lambda': 0.0008509490658518463, 'proj_dim': 80, 'hidden_channels': 94}

In [7]:
mlp_eval(X, y, reg_type="inner_mat", n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.81632656 0.79591835 0.875      0.77083331]
[0.76975524 0.81948215 0.80430157 0.8947096  0.77475158]
[0.7956044  0.81951139 0.78980158 0.87082516 0.77207543]
{'acc': 0.8107993125915527, 'f1_macro': 0.8126000300289199, 'f1_weighted': 0.8095635923557335, 'acc_std': 0.035192175733458814, 'f1_macro_std': 0.04500308927522579, 'f1_weighted_std': 0.034183905959255544}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5


| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |  
| MLP (mrna only) | 0.87 +/- 0.06 | 0.87 +/- 0.07 | 0.87 +/- 0.06 |  
| MLP (mrna only) | 0.87 +/- 0.05 | 0.86 +/- 0.06 | 0.87 +/- 0.05 |  

{'acc': 0.8560374259948731, 'f1_macro': 0.8531890166223708, 'f1_weighted': 0.8540459975191783, 'acc_std': 0.05063744215757385, 'f1_macro_std': 0.06194529762727774, 'f1_weighted_std': 0.049910632703314174}

mrna only study.best_value=2.607328324144426, study.best_params={'lr': 8.077069025405151e-05, 'l1_lambda': 0.0004400516634407218, 'l2_lambda': 5.624746318975069e-05, 'proj_dim': 145, 'hidden_channels': 243}

# Results

| Method | ACC | F1 | F1 Weighted |
| --- | --- | --- | --- |
| KNN | 0.80 +/- 0.04 | 0.79 +/- 0.04 | 0.79 +/- 0.04 |
| LIN SVM w RFE | 0.84 +/- 0.01 | 0.84 +/- 0.02 | 0.84 +/- 0.01 |
| XGBoost | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |
| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |
| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |
| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |
| MOGONET (mrna, cnv, mirna) | 0.766 ± 0.025 | 0.742 ± 0.027| 0.768 ± 0.024|
| MOGONET (mrna, meth, mirna) | 0.824 ± 0.029 | 0.812 ± 0.039| 0.824 ± 0.032|

In [133]:
# prepare 5 folds for cross validation
skf = StratifiedKFold(n_splits=5)

train_indices = []
val_indices = []
test_indices = []

for train_index, test_index in skf.split(X, y):
    train_indices.append(train_index)
    val_index, test_index = train_test_split(test_index, test_size=0.5, stratify=y[test_index])
    val_indices.append(val_index)
    test_indices.append(test_index)

best_val = torch.zeros(5, 3)
best_test = torch.zeros(5, 3)

In [150]:
mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=2000)

mrna_X_filt = torch.tensor(mrna_X[:, mrna_mask], dtype=torch.float32)

mrna_eps = 0.95
# meth_eps = 0.997
# cnv_eps = 0.9998
# mirna_eps = 0.962

# build adjacency matrix for each data type
A_mrna = cosine_similarity_matrix(mrna_X_filt)
A_mrna_bin = torch.where(A_mrna > mrna_eps, 1, 0) # - torch.eye(A_mrna.shape[0])
print(f"mrna mean degree {A_mrna_bin.sum(axis=1, dtype=torch.float32).mean()}")
print(
    f"Isolated samples = {(A_mrna_bin.sum(dim=1) == 1).sum()}"
)

mrna mean degree 32.97101593017578
Isolated samples = 131


In [108]:
from bipartite_gnn.graph_building import threshold_matrix

A_mrna = A_mrna.to(torch.float32)

threshold_matrix(A_mrna, self_connections=True, target_avg_degree=20)

0.75 tensor(482.8551)
0.875 tensor(370.6605)
0.9375 tensor(93.3520)
0.96875 tensor(2.1925)
0.953125 tensor(23.1408)
0.9609375 tensor(7.9689)
0.95703125 tensor(14.0062)
0.955078125 tensor(18.1180)
0.9541015625 tensor(20.3085)
Isolated samples = 0, avg degree = 20.308488845825195


tensor([[1., 1., 1.,  ..., 0., 0., 1.],
        [1., 1., 1.,  ..., 0., 0., 1.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [1., 1., 0.,  ..., 0., 0., 1.]])

In [161]:
for i in range(len(train_indices)):
    train_idx = train_indices[i]
    val_idx = val_indices[i]
    test_idx = test_indices[i]

    mrna_gene_names = mrna[:, 0].to_numpy()
    meth_gene_names = meth[:, 0].to_numpy()
    cna_gene_names = cnv[:, 0].to_numpy()
    mirna_gene_names = mirna[:, 0].to_numpy()

    mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=2000)
    meth_mask = class_variational_selection(meth_X[train_idx], y[train_idx], n_features=1000)
    cnv_mask = class_variational_selection(cnv_X[train_idx], y[train_idx], n_features=1000)
    mirna_mask = class_variational_selection(mirna_X[train_idx], y[train_idx], n_features=200)

    mrna_X_filt = mrna_X[:, mrna_mask]
    meth_X_filt = meth_X[:, meth_mask]
    cnv_X_filt = cnv_X[:, cnv_mask]
    mirna_X_filt = mirna_X[:, mirna_mask]

    # minmax scaling
    mms = MinMaxScaler()
    mms.fit(mrna_X_filt[train_idx])
    mrna_X_filt = torch.tensor(mms.transform(mrna_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(meth_X_filt[train_idx])
    meth_X_filt = torch.tensor(mms.transform(meth_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(cnv_X_filt[train_idx])
    cnv_X_filt = torch.tensor(mms.transform(cnv_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(mirna_X_filt[train_idx])
    mirna_X_filt = torch.tensor(mms.transform(mirna_X_filt), dtype=torch.float32)


    mrna_gene_names = mrna_gene_names[mrna_mask]
    meth_gene_names = meth_gene_names[meth_mask]
    cna_gene_names = cna_gene_names[cnv_mask]
    mirna_gene_names = mirna_gene_names[mirna_mask]

    ##### BUILDING GRAPH #####

    # build graph for mogonet
    data = pyg.data.HeteroData()

    mrna_eps = 0.933
    meth_eps = 0.99
    cnv_eps = 0.99
    mirna_eps = 0.962

    # build adjacency matrix for each data type
    A_mrna = cosine_similarity_matrix(mrna_X_filt)
    A_mrna_bin = torch.where(A_mrna > mrna_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    print(f"mrna mean degree {A_mrna_bin.sum(axis=1, dtype=torch.float32).mean()}")
    print(
        f"Isolated samples = {(A_mrna_bin.sum(dim=1) == 0).sum()}"
    )

    data['mrna'].x = mrna_X_filt
    data['mrna'].edge_index = dense_to_coo(A_mrna_bin)

    A_meth = cosine_similarity_matrix(meth_X_filt)
    A_meth_bin = torch.where(A_meth > meth_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_meth_bin.sum(axis=1, dtype=torch.float32).mean()
    print(f"meth mean degree {A_meth_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['meth'].x = meth_X_filt
    data['meth'].edge_index = dense_to_coo(A_meth_bin)

    A_cnv = cosine_similarity_matrix(cnv_X_filt)
    A_cnv_bin = torch.where(A_cnv > cnv_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_cnv_bin.sum(axis=1, dtype=torch.float32).mean() 
    print(f"cnv mean degree {A_cnv_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['cnv'].x = cnv_X_filt
    data['cnv'].edge_index = dense_to_coo(A_cnv_bin)

    A_mirna = cosine_similarity_matrix(mirna_X_filt)
    A_mirna_bin = torch.where(A_mirna > mirna_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_mirna_bin.sum(axis=1, dtype=torch.float32).mean()
    print(f"mirna mean degree {A_mirna_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['mirna'].x = mirna_X_filt
    data['mirna'].edge_index = dense_to_coo(A_mirna_bin)

    data.y = torch.tensor(y)

    data.train_mask = torch.tensor(train_idx)
    data.val_mask = torch.tensor(val_idx)
    data.test_mask = torch.tensor(test_idx)

    ##### TRAINING MODEL #####

    model = MOGONET(
        omics=data.x_dict.keys(),
        in_channels=[data.x_dict[omics].shape[1] for omics in data.x_dict.keys()],
        hidden_channels=300,
        num_classes=4,
        encoder_type="gat",
        dropout=0.0,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

    ccounts = torch.unique(data.y[data.train_mask], return_counts=True)[1]
    inv_w = 1.0 / ccounts.float()
    class_weights = inv_w / inv_w.sum()

    trainer = GNNTrainer(
        model=model,
        loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
        optimizer=optimizer,
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, min_lr=1e-5),
        params={
            "l1_lambda": None,
        }
    )

    bp = trainer.train(
        data=data,
        epochs=500,
        log_interval=50,
    )

    # print(bp)
    best_val[i] = bp[0]
    best_test[i] = bp[1]

mrna mean degree 10.786749839782715
Isolated samples = 257
meth mean degree 59.209110260009766
cnv mean degree 12.078675270080566
mirna mean degree 8.844720840454102
Epoch: 050:
Train Loss: 0.3424, Train Acc: 0.8394, Train F1 Macro: 0.8647, Train F1 Weighted: 0.8417
Val Loss: 0.5198, Val Acc: 0.7917, Val F1 Macro: 0.7911, Val F1 Weighted: 0.7990
Test Loss: 0.4351, Test Acc: 0.7959, Test F1 Macro: 0.8019, Test F1 Weighted: 0.8043
##################################################
Epoch: 100:
Train Loss: 0.0992, Train Acc: 0.9663, Train F1 Macro: 0.9729, Train F1 Weighted: 0.9663
Val Loss: 1.0371, Val Acc: 0.8750, Val F1 Macro: 0.8375, Val F1 Weighted: 0.8722
Test Loss: 0.3365, Test Acc: 0.8776, Test F1 Macro: 0.8685, Test F1 Weighted: 0.8800
##################################################
Epoch: 150:
Train Loss: 0.0783, Train Acc: 0.9715, Train F1 Macro: 0.9770, Train F1 Weighted: 0.9715
Val Loss: 1.2223, Val Acc: 0.8750, Val F1 Macro: 0.8375, Val F1 Weighted: 0.8722
Test Loss: 0.349

In [83]:
print(f"| MOGONET GCN (mrna only) | {best_test[:, 0].mean():.2f} +/- {best_test[:, 0].std():.2f} | {best_test[:, 1].mean():.2f} +/- {best_test[:, 1].std():.2f} | {best_test[:, 2].mean():.2f} +/- {best_test[:, 2].std():.2f} |")

| MOGONET GCN (mrna only) | 0.85 +/- 0.03 | 0.84 +/- 0.03 | 0.85 +/- 0.03 |


In [162]:
print(f"| MOGONET GAT (mrna only) | {best_val[:, 0].mean():.2f} +/- {best_val[:, 0].std():.2f} | {best_val[:, 1].mean():.2f} +/- {best_val[:, 1].std():.2f} | {best_val[:, 2].mean():.2f} +/- {best_val[:, 2].std():.2f} |")
print(f"| MOGONET GAT (mrna only) | {best_test[:, 0].mean():.2f} +/- {best_test[:, 0].std():.2f} | {best_test[:, 1].mean():.2f} +/- {best_test[:, 1].std():.2f} | {best_test[:, 2].mean():.2f} +/- {best_test[:, 2].std():.2f} |")

| MOGONET GAT (mrna only) | 0.87 +/- 0.05 | 0.87 +/- 0.05 | 0.87 +/- 0.05 |
| MOGONET GAT (mrna only) | 0.85 +/- 0.03 | 0.86 +/- 0.02 | 0.85 +/- 0.03 |


In [19]:
val, test = mogonet_eval(
    input_omics={
        "mrna": mrna,
        "meth": meth,
        "mirna": mirna,
    },
    n_input_features={
        "mrna": 3000,
        "meth": 1000,
        "mirna": 200,
    },
    y=y,
    params={
        "encoder_hidden_channels": {
            "mrna": 300,
            "meth": 100,
            "mirna": 50,
        },
        "encoder_type": "gat",
        "num_classes": 4,
        "graph_style": "threshold",
        "avg_degree": 15,
        # "knn": 15,
        "dropout": 0.0,
        "epochs": 400,
        "log_interval": 50,
        "save_best_model": False,
    }
)

dict_keys(['mrna', 'meth', 'mirna'])
Fold 0
Building graph for mrna
Building graph for meth
Building graph for mirna
Epoch: 050:
Train Loss: 0.3596, Train Acc: 0.8420, Train F1 Macro: 0.8590, Train F1 Weighted: 0.8425
Val Loss: 0.8048, Val Acc: 0.8333, Val F1 Macro: 0.7985, Val F1 Weighted: 0.8363
Test Loss: 0.6848, Test Acc: 0.7755, Test F1 Macro: 0.7683, Test F1 Weighted: 0.7748
##################################################
Epoch: 100:
Train Loss: 0.2584, Train Acc: 0.8860, Train F1 Macro: 0.8953, Train F1 Weighted: 0.8865
Val Loss: 1.0610, Val Acc: 0.8125, Val F1 Macro: 0.7948, Val F1 Weighted: 0.8173
Test Loss: 0.8789, Test Acc: 0.7347, Test F1 Macro: 0.6997, Test F1 Weighted: 0.7385
##################################################
Epoch: 150:
Train Loss: 0.2451, Train Acc: 0.8912, Train F1 Macro: 0.8993, Train F1 Weighted: 0.8917
Val Loss: 1.1161, Val Acc: 0.8125, Val F1 Macro: 0.7948, Val F1 Weighted: 0.8173
Test Loss: 0.9157, Test Acc: 0.7551, Test F1 Macro: 0.7243, Test 

In [20]:
print("MOGONET GAT (mrna, mirna, meth) results:")
print(f"| MOGONET GAT val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna, mirna, meth) results:
| MOGONET GAT val | 0.80 +/- 0.06 | 0.79 +/- 0.06 | 0.80 +/- 0.06 |
| MOGONET GAT test | 0.81 +/- 0.07 | 0.82 +/- 0.06 | 0.82 +/- 0.07 |


In [18]:
print("MOGONET GAT (mrna only) results:")
print(f"| MOGONET GAT (mrna only) val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT (mrna only) test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

| MOGONET GAT (mrna only) val | 0.80 +/- 0.06 | 0.81 +/- 0.05 | 0.81 +/- 0.05 |
| MOGONET GAT (mrna only) test | 0.86 +/- 0.04 | 0.84 +/- 0.05 | 0.86 +/- 0.04 |


# BIPARTITE GNN

In [ ]:
for i in range(len(train_indices)):
    train_idx = train_indices[i]
    val_idx = val_indices[i]
    test_idx = test_indices[i]

    mrna_gene_names = mrna[:, 0].to_numpy()
    meth_gene_names = meth[:, 0].to_numpy()
    cna_gene_names = cnv[:, 0].to_numpy()
    mirna_gene_names = mirna[:, 0].to_numpy()

    mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=3000)
    meth_mask = class_variational_selection(meth_X[train_idx], y[train_idx], n_features=500)
    cnv_mask = class_variational_selection(cnv_X[train_idx], y[train_idx], n_features=500)
    mirna_mask = class_variational_selection(mirna_X[train_idx], y[train_idx], n_features=200)

    mrna_X_filt = mrna_X[:, mrna_mask]
    meth_X_filt = meth_X[:, meth_mask]
    cnv_X_filt = cnv_X[:, cnv_mask]
    mirna_X_filt = mirna_X[:, mirna_mask]

    # minmax scaling
    mms = MinMaxScaler()
    mms.fit(mrna_X_filt[train_idx])
    mrna_X_filt = torch.tensor(mms.transform(mrna_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(meth_X_filt[train_idx])
    meth_X_filt = torch.tensor(mms.transform(meth_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(cnv_X_filt[train_idx])
    cnv_X_filt = torch.tensor(mms.transform(cnv_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(mirna_X_filt[train_idx])
    mirna_X_filt = torch.tensor(mms.transform(mirna_X_filt), dtype=torch.float32)


    mrna_gene_names = mrna_gene_names[mrna_mask]
    meth_gene_names = meth_gene_names[meth_mask]
    cna_gene_names = cna_gene_names[cnv_mask]
    mirna_gene_names = mirna_gene_names[mirna_mask]

    ##### BUILDING GRAPH #####

    # build graph for mogonet
    data = pyg.data.HeteroData()

    # prepare graph for birpartite gnn


    data['mirna'].x = mirna_X_filt
    data['mirna'].edge_index = dense_to_coo(A_mirna_bin)

    data.y = torch.tensor(y)

    data.train_mask = torch.tensor(train_idx)
    data.val_mask = torch.tensor(val_idx)
    data.test_mask = torch.tensor(test_idx)

    ##### TRAINING MODEL #####

    model = ...

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

    ccounts = torch.unique(data.y[data.train_mask], return_counts=True)[1]
    inv_w = 1.0 / ccounts.float()
    class_weights = inv_w / inv_w.sum()

    trainer = GNNTrainer(
        model=model,
        loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
        optimizer=optimizer,
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, min_lr=1e-5),
        params={
            "l1_lambda": None,
        }
    )

    bp = trainer.train(
        data=data,
        epochs=300,
        log_interval=50,
    )

    # print(bp)
    best_val[i] = bp[0]
    best_test[i] = bp[1]